# Optuna Hyperparameter Search — LSTM (PyTorch)

Grid-search the LSTM training hyperparameters that give the **lowest average validation MSE
across all portfolio tickers**, using Optuna's `GridSampler` over **16 combinations**
(4 LR × 4 batch size).

- **Fixed**: `patience=30`, `lr_plateau_patience=15` (not in search space)
- **Searched**: LR ∈ [1e-4, 5e-4, 1e-3, 5e-3] × batch ∈ [32, 64, 128, 256]
- **Objective**: mean val MSE across **all** universe tickers — single study, aggregate
- **Auto-applied**: best params saved to `output/best_lstm_params.json`, patched into
  `src/constants.py`, and loaded automatically by `BL_LSTM_Portfolio.ipynb` at startup.

Adapted from `Optuna_Hyperparameter_Search_Plan.md`. Reuses `src/` modules:

| Plan (Keras)                         | Here (this repo)                                   |
|--------------------------------------|----------------------------------------------------|
| `build_lstm()` Keras `Sequential`    | `src.lstm_model.LSTMRegressor` (4×60, Arch B)      |
| `model.fit(...)` + Keras callbacks   | `src.lstm_model._train_one` (early-stop + LR plateau) |
| load `data/processed/{t}_train.npz`  | `fetch_data.fetch` → `build_stock_dataset` (cache-backed) |
| `TFKerasPruningCallback`             | omitted — GridSampler runs all 16 exhaustively     |
| `results/optuna_mse_map.png`         | `output/optuna_mse_map.png`                        |

Open this notebook **from the `bl_lstm_views/` folder** so `from src import …` resolves.

In [1]:
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import optuna

from src import constants as C
from src import fetch_data
from src.preprocessing import build_stock_dataset
from src.lstm_model import _train_one, get_device, _empty_cache

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)   # silence per-trial optuna noise

# ── Universe — same as BL_LSTM_Portfolio ─────────────────────────────────────
TICKERS = ["RKLB", "AMAT", "CRWD", "IONQ", "LLY", "EOSE", "ASML", "ELV", "PLTR", "OKLO", "NVDA", "MSFT"]
DEVICE  = get_device(os.environ.get("OPTUNA_DEVICE", "auto"))

# ── Patience: fixed (not searched) ───────────────────────────────────────────
FIXED_PATIENCE    = 30    # matches C.EARLY_STOP_PATIENCE
FIXED_LR_PATIENCE = 15    # matches C.LR_PLATEAU_PATIENCE
FIXED_EPOCHS      = 600   # early stopping will end runs well before this

# ── Search: LR × batch only  →  4 × 4 = 16 combinations ─────────────────────
SEARCH_SPACE = {
    "learning_rate": [1e-4, 5e-4, 1e-3, 5e-3],
    "batch_size":    [32, 64, 128, 256],
}
N_TRIALS = int(np.prod([len(v) for v in SEARCH_SPACE.values()]))  # = 16

print(f"device={DEVICE.type}  fixed_epochs={FIXED_EPOCHS}  "
      f"patience={FIXED_PATIENCE}/{FIXED_LR_PATIENCE}  trials={N_TRIALS}")

/Users/doyinratnavardhana/Quantitative-modeler/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=mps  fixed_epochs=600  patience=30/15  trials=16


## Step 1 — Load data

No `data/processed/*.npz` in this repo — datasets are built in-memory exactly like the main
pipeline: `fetch_data.fetch` (cache-backed) → `build_stock_dataset` (8 indicators, train-only
MinMax, 30-day windows, 5-day-ahead targets). A ticker whose data won't fetch or is too short
to window is **skipped**, and the search continues on the rest.

In [2]:
raw = fetch_data.fetch(TICKERS, with_market_caps=False)
datasets = {}
for t, df in raw["prices"].items():
    try:
        sd = build_stock_dataset(t, df)
        w = sd.windows
        datasets[t] = {
            "X_train": w["X"][w["train"]], "Yf_train": w["Y"][w["train"]],
            "X_val":   w["X"][w["val"]],   "Yf_val":   w["Y"][w["val"]],
        }
        print(f"✅ {t:<5} loaded — train: {datasets[t]['X_train'].shape}, "
              f"val: {datasets[t]['X_val'].shape}")
    except Exception as e:
        print(f"⚠️  {t}: skipped ({e})")

TICKERS = list(datasets.keys())   # reconcile to what actually built
assert TICKERS, "no usable tickers"


  ↗  Fetching ['RKLB', 'AMAT', 'CRWD', 'IONQ', 'LLY', 'EOSE', 'ASML', 'ELV', 'PLTR', 'OKLO', 'NVDA', 'MSFT']  (2010-01-01 → 2026-06-24)
      (minimum 878 days required: 500 train + 126 val + 252 test)
  ✅  RKLB: 1399 daily bars
  ✅  AMAT: 4142 daily bars
  ✅  CRWD: 1767 daily bars
  ✅  IONQ: 1373 daily bars
  ✅  LLY: 4142 daily bars
  ✅  EOSE: 1415 daily bars
  ✅  ASML: 4142 daily bars
  ✅  ELV: 4142 daily bars
  ✅  PLTR: 1438 daily bars
  ✅  OKLO: 1245 daily bars
  ✅  NVDA: 4142 daily bars
  ✅  MSFT: 4142 daily bars
✅ RKLB  loaded — train: (962, 30, 8), val: (126, 30, 8)
✅ AMAT  loaded — train: (3705, 30, 8), val: (126, 30, 8)
✅ CRWD  loaded — train: (1330, 30, 8), val: (126, 30, 8)
✅ IONQ  loaded — train: (936, 30, 8), val: (126, 30, 8)
✅ LLY   loaded — train: (3705, 30, 8), val: (126, 30, 8)
✅ EOSE  loaded — train: (978, 30, 8), val: (126, 30, 8)
✅ ASML  loaded — train: (3705, 30, 8), val: (126, 30, 8)
✅ ELV   loaded — train: (3705, 30, 8), val: (126, 30, 8)
✅ PLTR  loaded — train:

## Step 2 — Trainer

The plan's `build_lstm()` + Keras `model.fit` are replaced by `src.lstm_model._train_one`,
which builds the same 4×60 multi-output `LSTMRegressor` (Architecture B) and trains it with
**val-driven early stopping + ReduceLROnPlateau**, returning the best validation MSE — exactly
the quantity Optuna minimises. (`epochs` is the *cap*; early-stopping may end a trial sooner.)

In [3]:
def trial_val_mse(ticker, lr, bs):
    """Train one LSTM with the given hyperparameters; return its best validation MSE."""
    d = datasets[ticker]
    torch.manual_seed(42)   # same init across trials → fair hyperparameter comparison
    model, best_val = _train_one(
        d["X_train"], d["Yf_train"], d["X_val"], d["Yf_val"],
        epochs=FIXED_EPOCHS, lr=lr, batch=bs, device=DEVICE, ticker=ticker, log_every=0,
        patience=FIXED_PATIENCE, lr_plateau_patience=FIXED_LR_PATIENCE,
    )
    del model
    _empty_cache(DEVICE)
    return float(best_val)

## Step 3 — Optuna objective

`make_objective()` trains every stock in the universe with the suggested hyperparameters
and returns the **mean val MSE across all tickers** as a single scalar for Optuna to
minimise. One study covers the entire universe — this avoids per-stock meta-overfitting
and ensures the chosen LR/batch generalises across the portfolio.

In [4]:
def make_objective():
    """Single objective: mean val MSE across all loaded tickers."""
    def objective(trial):
        lr = trial.suggest_categorical("learning_rate", SEARCH_SPACE["learning_rate"])
        bs = trial.suggest_categorical("batch_size",    SEARCH_SPACE["batch_size"])
        mse_list = []
        for ticker in TICKERS:
            if ticker not in datasets:
                continue
            try:
                mse_list.append(trial_val_mse(ticker, lr, bs))
            except Exception as e:
                print(f"  ⚠️  {ticker} failed (lr={lr}, bs={bs}): {e}")
        if not mse_list:
            raise RuntimeError("no tickers produced valid MSE")
        return float(np.mean(mse_list))
    return objective

## Step 4 — Run the search

One `GridSampler` study with **16 trials** (4 LR × 4 batch). Each trial trains every
universe stock and reports the aggregate mean val MSE. Best params are auto-saved to
`output/best_lstm_params.json` and patched into `src/constants.py`.

In [ ]:
import re

def _trial_callback(study, trial):
    if trial.value is None:
        return
    best_mark = "  ⭐ best!" if trial.value <= study.best_value + 1e-12 else ""
    p = trial.params
    print(f"  Trial {trial.number + 1:>2}/{N_TRIALS} | "
          f"lr={p['learning_rate']:.4f}  bs={p['batch_size']:>3} "
          f"→ mean_MSE={trial.value:.4f}{best_mark}")

print("=" * 58)
print(f"🔍 Aggregate search  ({N_TRIALS} combinations × {len(TICKERS)} stocks each)")
print("=" * 58)

study = optuna.create_study(
    direction="minimize",
    study_name="lstm_aggregate",
    sampler=optuna.samplers.GridSampler(SEARCH_SPACE),
)
study.optimize(make_objective(), n_trials=N_TRIALS, callbacks=[_trial_callback])

best = dict(study.best_params)
best["val_mse_mean"] = study.best_value

# ── auto-save best params to JSON ────────────────────────────────────────────
os.makedirs(C.OUTPUT_DIR, exist_ok=True)
params_path = os.path.join(C.OUTPUT_DIR, "best_lstm_params.json")
json.dump(best, open(params_path, "w"), indent=2)
print(f"\n✅ best → lr={best['learning_rate']}, batch={best['batch_size']}, "
      f"mean_val_MSE={best['val_mse_mean']:.4f}")
print(f"   saved → {os.path.relpath(params_path, C.ROOT_DIR)}")

# ── auto-patch src/constants.py ──────────────────────────────────────────────
constants_path = os.path.join(C.ROOT_DIR, "src", "constants.py")
with open(constants_path) as f:
    content = f.read()
content = re.sub(r"(LSTM_LR\s*=\s*)[\d.e+\-]+", f"\\g<1>{best['learning_rate']}", content)
content = re.sub(r"(LSTM_BATCH\s*=\s*)\d+",        f"\\g<1>{best['batch_size']}", content)
with open(constants_path, "w") as f:
    f.write(content)
print(f"   constants.py patched: LSTM_LR={best['learning_rate']}, LSTM_BATCH={best['batch_size']}")
print("\n⚠️  Restart BL_LSTM_Portfolio.ipynb kernel to pick up the new constants.py values,")
print("   OR it will auto-load from best_lstm_params.json if the JSON path is found (see Cell 1 there).")

🔍 Aggregate search  (16 combinations × 12 stocks each)
      [RKLB] early stop at epoch 80 (best val 0.1185)
      [AMAT] early stop at epoch 105 (best val 0.0258)
      [CRWD] early stop at epoch 100 (best val 0.0871)
      [IONQ] early stop at epoch 190 (best val 0.1439)
      [LLY] early stop at epoch 87 (best val 0.0562)
      [EOSE] early stop at epoch 81 (best val 0.0721)
      [ASML] early stop at epoch 134 (best val 0.0318)


## Step 5 — MSE results

All 16 combinations, sorted best → worst (aggregate mean val MSE), plus a heatmap
(lr × batch, colour = mean MSE). The best combination is marked ⭐ and highlighted in gold.

In [ ]:
trials = [t for t in study.trials if t.value is not None]
trials.sort(key=lambda t: t.value)

print("\n" + "=" * 56)
print("📊 Aggregate val MSE — best → worst  (all tickers, mean)")
print("=" * 56)
print(f"  {'#':>3}  {'LR':>8}  {'Batch':>5}  {'Mean Val MSE':>13}")
print("  " + "-" * 38)
for rank, t in enumerate(trials, 1):
    star = "⭐" if rank == 1 else "  "
    p = t.params
    print(f"  {star}{rank:>2}.  {p['learning_rate']:>8.4f}  {p['batch_size']:>5}  {t.value:>13.4f}")

# ── heatmap: lr (log x) × batch (y), colour = mean MSE ──────────────────────
fig, ax = plt.subplots(figsize=(7.2, 4.5))
sc = ax.scatter(
    [t.params["learning_rate"] for t in trials],
    [t.params["batch_size"]    for t in trials],
    c=[t.value for t in trials],
    s=280, cmap="RdYlGn_r", alpha=0.85, edgecolors="k", linewidths=0.5,
)
plt.colorbar(sc, ax=ax, label="Mean Val MSE (all tickers)")
ax.set_xscale("log")
ax.set_xlabel("Learning rate")
ax.set_ylabel("Batch size")
ax.set_yticks(SEARCH_SPACE["batch_size"])
ax.set_title("Aggregate LSTM search — mean val MSE across all portfolio tickers",
             fontweight="bold")
best_t = trials[0]
ax.scatter(best_t.params["learning_rate"], best_t.params["batch_size"],
           marker="*", s=450, color="gold", edgecolors="k", linewidths=0.7, zorder=5,
           label=f"best: lr={best_t.params['learning_rate']}, bs={best_t.params['batch_size']}\nMSE={best_t.value:.4f}")
ax.legend(fontsize=9, loc="best")
plt.tight_layout()
out_png = os.path.join(C.OUTPUT_DIR, "optuna_mse_map.png")
os.makedirs(C.OUTPUT_DIR, exist_ok=True)
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n📈 saved → {os.path.relpath(out_png, C.ROOT_DIR)}")